# CONFIRM 5 — Where does the anchored beta actually start responding to lmin?

Purpose: the fiducial check only tested lmin = 51, 131, 211, where sigma_beta moved by
just ~0.1% (0.1191 -> 0.1192 -> 0.1193 deg) -- real, but tiny, and invisible at the
3-decimal precision used in the paper/slide. The working hypothesis is that this is
because the anchor uncertainty (~0.12-0.14 deg per map, fixed, from the full-range MK
run) dominates the error budget over the CMB-derived theta uncertainty (~0.06-0.08 deg
per map), so even a real fractional increase in sigma_theta barely moves the combined
sigma_beta -- until sigma_theta grows enough (from cutting a much larger chunk of the
ell range) to matter.

This notebook pushes lmin much further out (51 up to ~1291, i.e. cutting away most of
the ell range while lmax stays fixed near 1490), holding the anchor fixed throughout
(exactly as in the fiducial check), and tracks:
  - per-map sigma_theta(lmin)  -- should grow as lmin increases (fewer modes retained)
  - combined sigma_beta(lmin)  -- should stay ~flat while sigma_theta << sigma_anchor,
    then visibly increase once sigma_theta becomes comparable to / larger than sigma_anchor

This directly tests whether "the fit is really not changing" (my earlier concern) vs.
"the fit changes correctly, but is swamped by a fixed, larger anchor uncertainty until
you remove enough data" (the current hypothesis).

Uses the existing mask-0 MK chain and saved spectra only; refits theta at each lmin
in the notebook (this is real recomputation, not cached results). Prints a `RESULT-5`
block and saves a diagnostic figure; copy the RESULT-5 block back verbatim.


In [1]:
import os
for v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[v]='1'
%cd /global/homes/l/lonappan/workspace/cosmic_birefringence
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from cosmic_bire.anchored import AnchoredBeta
from cosmic_bire.anchored_theta import run, load_mk_chain, FREQS

config = 'configs/planck_hfi.yml'


/global/u2/l/lonappan/workspace/cosmic_birefringence


In [2]:
# --- fixed anchor, computed ONCE from the full-range mask-0 MK run ---
# (identical to the fiducial check: only the theta fit below varies with lmin)
res = run(config, freqs=FREQS, verbose=False)
anchor_mean, anchor_cov = res['anchor_mean'], res['anchor_cov']
labels = res['mask0']['labels']   # e.g. ['100A','143A','100B','143B']
print('Anchor (fixed for this whole notebook):')
for k, lbl in enumerate(labels):
    print(f'  {lbl}: {anchor_mean[k]:+.4f} +/- {np.sqrt(anchor_cov[k,k]):.4f} deg')
LMAX_DEFAULT = 1491
BIN_WIDTH = 20


Using existing /global/homes/l/lonappan/pscratch/CBDATA/theory/beam_corrected_lcdm_spectra_100_143_217_353.npy
Using existing /global/homes/l/lonappan/pscratch/CBDATA/theory/psi_l_sigma15.npy
Number of alphas: 8
Using hfi/lfi ps mask with CO. Index: 0
Importing observed cl. Filename: /global/homes/l/lonappan/pscratch/CBDATA/spectra/raw/cl_mask_percent_0_freq_100_143_217_353.npy
Importing beam smoothed LCDM cl. Filename: /global/homes/l/lonappan/pscratch/CBDATA/theory/beam_corrected_lcdm_spectra_100_143_217_353.npy
Loading /global/homes/l/lonappan/pscratch/CBDATA/theory/psi_l_sigma15.npy
Loading saved covariance matrix. Filename: /global/homes/l/lonappan/pscratch/CBDATA/covariance/cov_bin_cl_hfi_mask_percent_0_100_143_217_353_lmin51_lmax1491.npy
Loaded 218368 samples directly from /global/homes/l/lonappan/pscratch/CBDATA/chains/planck_hfi_mask_0.h5
Removed no burn in


/global/u2/l/lonappan/workspace/cosmic_birefringence/src/cosmic_bire/tools_fast.py:170: NumbaPerformanceWarning: np.dot() is faster on contiguous arrays, called on (Array(float64, 2, 'C', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  C = np.dot(np.dot(A, obs_cov), np.transpose(A))
/global/u2/l/lonappan/workspace/cosmic_birefringence/src/cosmic_bire/tools_fast.py:170: NumbaPerformanceWarning: np.dot() is faster on contiguous arrays, called on (Array(float64, 2, 'C', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  C = np.dot(np.dot(A, obs_cov), np.transpose(A))


Anchor (fixed for this whole notebook):
  100A: -0.3264 +/- 0.1355 deg
  143A: +0.0158 +/- 0.1162 deg
  100B: -0.4316 +/- 0.1318 deg
  143B: +0.1358 +/- 0.1166 deg


In [3]:
# --- lmin grid: bin-aligned (51 + 20n), pushed well beyond the fiducial 51-211 range ---
lmin_grid = [51, 131, 211, 311, 411, 511, 611, 711, 811, 911, 1011, 1111, 1211, 1291]

def lmax_for(lmin, lmax_default=LMAX_DEFAULT, width=BIN_WIDTH):
    # largest lmax <= lmax_default such that (lmax-lmin) is a multiple of width
    resid = (lmax_default - lmin) % width
    return lmax_default if resid == 0 else lmax_default - resid

rows = []
for lmin in lmin_grid:
    lmax_use = lmax_for(lmin)
    n_bins = (lmax_use - lmin) // BIN_WIDTH
    if n_bins < 3:
        print(f'lmin={lmin}: only {n_bins} bins left, skipping (too few for a stable fit)')
        continue
    ab = AnchoredBeta(config, mask='0', freqs=FREQS, lmin=lmin, lmax=lmax_use)
    theta, theta_cov = ab.fit_theta()
    beta, sigma_beta, beta_per_map, sigma_per_map = ab.beta_profile(anchor_mean, anchor_cov, theta, theta_cov)
    sigma_theta = np.sqrt(np.diag(theta_cov))
    rows.append(dict(lmin=lmin, lmax=lmax_use, n_bins=n_bins,
                      beta=float(beta), sigma_beta=float(sigma_beta),
                      sigma_theta=sigma_theta.copy(), theta=theta.copy()))
    print(f'lmin={lmin:5d}  lmax={lmax_use:5d}  n_bins={n_bins:3d}  '
          f'beta={beta:+.4f}+/-{sigma_beta:.4f}  '
          f'sigma_theta=' + ', '.join(f'{lbl}:{s:.4f}' for lbl, s in zip(labels, sigma_theta)))


lmin=   51  lmax= 1491  n_bins= 72  beta=+0.3616+/-0.1191  sigma_theta=100A:0.0825, 143A:0.0570, 100B:0.0739, 143B:0.0559
lmin=  131  lmax= 1491  n_bins= 68  beta=+0.3595+/-0.1192  sigma_theta=100A:0.0895, 143A:0.0631, 100B:0.0799, 143B:0.0614
lmin=  211  lmax= 1491  n_bins= 64  beta=+0.3596+/-0.1193  sigma_theta=100A:0.0925, 143A:0.0656, 100B:0.0822, 143B:0.0638
lmin=  311  lmax= 1491  n_bins= 59  beta=+0.3474+/-0.1198  sigma_theta=100A:0.0985, 143A:0.0696, 100B:0.0870, 143B:0.0675
lmin=  411  lmax= 1491  n_bins= 54  beta=+0.3333+/-0.1237  sigma_theta=100A:0.1287, 143A:0.0873, 100B:0.1117, 143B:0.0842
lmin=  511  lmax= 1491  n_bins= 49  beta=+0.3729+/-0.1278  sigma_theta=100A:0.1575, 143A:0.1032, 100B:0.1350, 143B:0.0995
lmin=  611  lmax= 1491  n_bins= 44  beta=+0.3835+/-0.1303  sigma_theta=100A:0.1734, 143A:0.1124, 100B:0.1479, 143B:0.1081
lmin=  711  lmax= 1491  n_bins= 39  beta=+0.3256+/-0.1479  sigma_theta=100A:0.2610, 143A:0.1621, 100B:0.2186, 143B:0.1566
lmin=  811  lmax= 1491  

In [4]:
# --- where does sigma_beta actually start moving? ---
base_sigma = rows[0]['sigma_beta']
print(f'baseline (lmin={rows[0]["lmin"]}) sigma_beta = {base_sigma:.4f} deg\n')
print(f'{"lmin":>6} {"sigma_beta":>11} {"rel. change":>12}')
thresholds = [0.05, 0.10, 0.20]
first_cross = {t: None for t in thresholds}
for r in rows:
    rel = r['sigma_beta']/base_sigma - 1.0
    print(f'{r["lmin"]:6d} {r["sigma_beta"]:11.4f} {rel*100:11.2f}%')
    for t in thresholds:
        if first_cross[t] is None and rel >= t:
            first_cross[t] = r['lmin']
print()
for t in thresholds:
    lm = first_cross[t]
    print(f'sigma_beta first exceeds baseline by {t*100:.0f}% at lmin = {lm}' if lm is not None
          else f'sigma_beta never reaches +{t*100:.0f}% within the tested range (up to lmin={rows[-1]["lmin"]})')


baseline (lmin=51) sigma_beta = 0.1191 deg

  lmin  sigma_beta  rel. change
    51      0.1191        0.00%
   131      0.1192        0.06%
   211      0.1193        0.13%
   311      0.1198        0.61%
   411      0.1237        3.82%
   511      0.1278        7.31%
   611      0.1303        9.37%
   711      0.1479       24.20%
   811      0.1730       45.20%
   911      0.1852       55.45%
  1011      0.2685      125.39%
  1111      0.4849      307.09%
  1211      0.5763      383.76%
  1291      0.8347      600.73%

sigma_beta first exceeds baseline by 5% at lmin = 511
sigma_beta first exceeds baseline by 10% at lmin = 711
sigma_beta first exceeds baseline by 20% at lmin = 711


In [6]:
# --- diagnostic figure: per-map sigma_theta and combined sigma_beta vs lmin ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

lmins = [r['lmin'] for r in rows]
sig_theta_arr = np.array([r['sigma_theta'] for r in rows])  # (n_lmin, n_maps)
for k, lbl in enumerate(labels):
    axes[0].plot(lmins, sig_theta_arr[:, k], marker='o', ms=3, label=lbl)
axes[0].set_xlabel(r'$\ell_{\min}$'); axes[0].set_ylabel(r'$\sigma_\theta$ [deg]')
axes[0].set_title('Per-map total-rotation uncertainty'); axes[0].legend(frameon=False, fontsize=8)

sig_beta_arr = np.array([r['sigma_beta'] for r in rows])
axes[1].plot(lmins, sig_beta_arr, marker='o', color='k')
axes[1].axhline(base_sigma, ls='--', color='0.6', lw=0.8)
for k, lbl in enumerate(labels):
    axes[1].axhline(np.sqrt(anchor_cov[k,k]), ls=':', lw=0.6, color='C'+str(k))
axes[1].set_xlabel(r'$\ell_{\min}$'); axes[1].set_ylabel(r'$\sigma_\beta$ [deg]')
axes[1].set_title('Combined anchored uncertainty\n(dotted = fixed per-map anchor sigma)')

fig.tight_layout()
plt.show()



In [7]:
print('==================== RESULT-5 ====================')
print('ANCHOR (fixed):')
for k, lbl in enumerate(labels):
    print(f'  {lbl}: {anchor_mean[k]:+.4f} +/- {np.sqrt(anchor_cov[k,k]):.4f} deg')
print()
print('LMIN_SCAN (lmin, lmax, n_bins, beta, sigma_beta, sigma_theta_per_map):')
for r in rows:
    st = ','.join(f'{lbl}={s:.4f}' for lbl, s in zip(labels, r['sigma_theta']))
    print(f'  lmin={r["lmin"]} lmax={r["lmax"]} n_bins={r["n_bins"]} '
          f'beta={r["beta"]:+.4f} sigma_beta={r["sigma_beta"]:.4f} sigma_theta=[{st}]')
print()
for t in thresholds:
    lm = first_cross[t]
    print(f'FIRST_LMIN_EXCEEDING_{int(t*100)}PCT: {lm}')
print('=================================================')


==================== RESULT-5 ====================
ANCHOR (fixed):
  100A: -0.3264 +/- 0.1355 deg
  143A: +0.0158 +/- 0.1162 deg
  100B: -0.4316 +/- 0.1318 deg
  143B: +0.1358 +/- 0.1166 deg

LMIN_SCAN (lmin, lmax, n_bins, beta, sigma_beta, sigma_theta_per_map):
  lmin=51 lmax=1491 n_bins=72 beta=+0.3616 sigma_beta=0.1191 sigma_theta=[100A=0.0825,143A=0.0570,100B=0.0739,143B=0.0559]
  lmin=131 lmax=1491 n_bins=68 beta=+0.3595 sigma_beta=0.1192 sigma_theta=[100A=0.0895,143A=0.0631,100B=0.0799,143B=0.0614]
  lmin=211 lmax=1491 n_bins=64 beta=+0.3596 sigma_beta=0.1193 sigma_theta=[100A=0.0925,143A=0.0656,100B=0.0822,143B=0.0638]
  lmin=311 lmax=1491 n_bins=59 beta=+0.3474 sigma_beta=0.1198 sigma_theta=[100A=0.0985,143A=0.0696,100B=0.0870,143B=0.0675]
  lmin=411 lmax=1491 n_bins=54 beta=+0.3333 sigma_beta=0.1237 sigma_theta=[100A=0.1287,143A=0.0873,100B=0.1117,143B=0.0842]
  lmin=511 lmax=1491 n_bins=49 beta=+0.3729 sigma_beta=0.1278 sigma_theta=[100A=0.1575,143A=0.1032,100B=0.1350,143B=0.